In [ ]:
import datetime
import json
import re
import time

from pathlib import Path

import pandas as pd

from rapidfuzz import fuzz

from googleapiclient.discovery import build

In [ ]:
API_KEY = "YOUR_API_KEY"

youtube = build(
    "youtube",
    "v3",
    developerKey=API_KEY
)

CHANNEL_REGISTRY = [
    {
        "id": "UCqZQlzSHbVJrwrn5XvzrzcA",
        "name": "NBC Sports"
    },
    {
        "id": "UCu7phdCr-raU7OaJfEpHZww",
        "name": "GCN Racing"
    },
    {
        "id": "UCfDfvvMARk4TKcC62ALi6eA",
        "name": "TNT Sports Cycling"
    },
    {
        "id": "UCuTaETsuCOkJ0H_GAztWt0Q",
        "name": "GCN"
    }
]

CACHE_DIR = Path("video_cache")
CACHE_DIR.mkdir(exist_ok=True)

CACHE_FILE = CACHE_DIR / "all_channel_videos.parquet"

In [ ]:
def get_upload_playlist(channel_id):

    resp = youtube.channels().list(
        part="contentDetails",
        id=channel_id
    ).execute()

    items = resp.get("items", [])

    if not items:
        return None

    return items[0]["contentDetails"]["relatedPlaylists"]["uploads"]

In [ ]:
def fetch_channel_videos(channel):

    playlist_id = get_upload_playlist(channel["id"])

    rows = []

    next_page = None

    while True:

        resp = youtube.playlistItems().list(
            part="snippet",
            playlistId=playlist_id,
            maxResults=50,
            pageToken=next_page
        ).execute()

        for item in resp["items"]:

            snippet = item["snippet"]

            rows.append({
                "video_id":
                    snippet["resourceId"]["videoId"],

                "title":
                    snippet["title"],

                "published":
                    snippet["publishedAt"],

                "channel":
                    channel["name"]
            })

        next_page = resp.get("nextPageToken")

        if not next_page:
            break

    return pd.DataFrame(rows)

In [ ]:
all_dfs = []

for channel in CHANNEL_REGISTRY:

    print(f"Downloading {channel['name']}")

    df = fetch_channel_videos(channel)

    all_dfs.append(df)

all_videos = pd.concat(
    all_dfs,
    ignore_index=True
)

all_videos.to_parquet(CACHE_FILE)

print(len(all_videos))

In [ ]:
all_videos = pd.read_parquet(CACHE_FILE)

all_videos["published"] = pd.to_datetime(
    all_videos["published"]
)

In [ ]:
def score_local_video(
    row,
    race_name,
    race_date,
    stage=None
):

    score = 0

    title = row["title"].lower()

    race_dt = pd.to_datetime(race_date)

    pub_dt = pd.to_datetime(row["published"])

    days_diff = abs(
        (pub_dt.date() - race_dt.date()).days
    )

    if days_diff == 0:
        score += 60
    elif days_diff <= 1:
        score += 50
    elif days_diff <= 3:
        score += 35
    elif days_diff <= 7:
        score += 15

    name_score = fuzz.partial_ratio(
        race_name.lower(),
        title
    )

    score += name_score * 0.25

    if name_score < 40:
        score -= 40

    if stage:

        if any(
            x in title
            for x in [
                f"stage {stage}",
                f"stage{stage}",
                f"étape {stage}",
                f"tappa {stage}"
            ]
        ):
            score += 30
        else:
            score -= 10

    if str(race_dt.year) in title:
        score += 15

    if "extended" in title:
        score += 10

    elif "highlights" in title:
        score += 5

    return score

In [ ]:
def find_best_video_local(
    race_name,
    race_date,
    stage=None
):

    candidates = all_videos.copy()

    race_year = str(race_date)[:4]

    candidates = candidates[
        candidates["title"]
            .str.contains(
                race_year,
                case=False,
                na=False
            )
    ]

    scores = []

    for _, row in candidates.iterrows():

        score = score_local_video(
            row,
            race_name,
            race_date,
            stage
        )

        scores.append(score)

    candidates["match_score"] = scores

    candidates = candidates.sort_values(
        "match_score",
        ascending=False
    )

    if len(candidates) == 0:
        return {"found": False}

    best = candidates.iloc[0]

    return {
        "found": True,
        "video_id": best["video_id"],
        "title": best["title"],
        "published": str(best["published"]),
        "channel": best["channel"],
        "url":
            f"https://youtube.com/watch?v={best['video_id']}",
        "match_score": float(best["match_score"]),
        "all_candidates":
            candidates.head(10).to_dict("records")
    }

In [ ]:
video = find_best_video_local(
    race_name,
    race_date,
    stage
)